WEBSCRAPPING


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin

# URLs de las 6 subcategorías
urls_categorias = {
    "PC": "https://www.pccomponentes.com/ordenadores-gaming",
    "Monitores": "https://www.pccomponentes.com/monitores-gaming",
    "Teclados": "https://www.pccomponentes.com/teclados-gaming",
    "Ratones": "https://www.pccomponentes.com/ratones-gaming",
    "Cascos": "https://www.pccomponentes.com/auriculares-gaming",
    "Mandos": "https://www.pccomponentes.com/mandos-pc",
}

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
}

def limpiar_precio(valor):
    if not valor:
        return None
    valor = valor.replace(".", "").replace(",", ".").replace("€", "").strip()
    try:
        return float(re.search(r"\d+(\.\d+)?", valor).group())
    except:
        return None

def limpiar_descuento(valor):
    if not valor:
        return None
    m = re.search(r"-?(\d+)", valor)
    return float(m.group(1)) if m else None

def limpiar_rating(valor):
    if not valor:
        return None
    valor = valor.replace(",", ".").strip()
    m = re.search(r"\d+(\.\d+)?", valor)
    return float(m.group()) if m else None

def limpiar_opiniones(valor):
    if not valor:
        return None
    m = re.search(r"\d+", valor.replace(".", ""))
    return int(m.group()) if m else None

def get_page_html(url):
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    return resp.text

def parse_product_list(html, subcategoria_manual, url_base):
    soup = BeautifulSoup(html, "html.parser")
    product_links = soup.select("a[data-product-name][data-product-id]")

    print(f"  → {len(product_links)} productos encontrados en {subcategoria_manual}")

    data = []

    for link in product_links:
        name = link.get("data-product-name")
        brand = link.get("data-product-brand")
        category = link.get("data-product-category")
        price_str = link.get("data-product-price")
        discount_pct_str = link.get("data-product-total-discount")
        sku = link.get("data-product-id")
        href = link.get("href")

        product_url = urljoin(url_base, href) if href else None

        card = link.find_parent("div", class_=re.compile(r"container"))
        rating_text = None
        reviews_text = None
        original_price_text = None
        seller_text = None
        promo_text = None

        if card:
            crossed = card.select_one("[data-e2e-crossed-price]")
            if crossed:
                original_price_text = crossed.get_text(strip=True)

            rating_block = card.select_one('[class*="ratingTypeOne"]')
            if rating_block:
                spans = rating_block.select("span")
                if len(spans) >= 1:
                    rating_text = spans[0].get_text(strip=True)
                if len(spans) >= 2:
                    reviews_text = spans[1].get_text(strip=True)

            seller_candidate = card.find(string=re.compile(r"Vendido y enviado por|Vendido por", re.I))
            if seller_candidate:
                seller_text = seller_candidate.strip()

            promo_candidate = card.find(string=re.compile(r"Trending|Top ventas|Aniversario|Nominado", re.I))
            if promo_candidate:
                promo_text = promo_candidate.strip()

        data.append({
            "sku": sku,
            "nombre": name,
            "marca": brand,
            "subcategoria": subcategoria_manual,
            "categoria_raw": category,
            "precio_actual_raw": price_str,
            "precio_original_raw": original_price_text,
            "descuento_pct_raw": discount_pct_str,
            "rating_raw": rating_text,
            "opiniones_raw": reviews_text,
            "seller_raw": seller_text,
            "promo_raw": promo_text,
            "product_url": product_url,
            "url_categoria": url_base,

            "precio_actual": limpiar_precio(price_str),
            "precio_original": limpiar_precio(original_price_text),
            "descuento_pct": limpiar_descuento(discount_pct_str),
            "rating": limpiar_rating(rating_text),
            "num_opiniones": limpiar_opiniones(reviews_text),
        })

    return data

all_productos = []

for subcategoria, url in urls_categorias.items():
    print(f"Scrapeando: {subcategoria} ({url})")
    try:
        html = get_page_html(url)
        productos = parse_product_list(html, subcategoria, url)
        all_productos.extend(productos)
    except Exception as e:
        print(f"  ❌ Error en {subcategoria}: {e}")

    time.sleep(2)

df = pd.DataFrame(all_productos)

print(f"\n✅ Total de productos scrapeados: {len(df)}")
if not df.empty:
    print(f"✅ Subcategorías únicas: {df['subcategoria'].unique()}")
    print("\nPrimeras filas:")
    print(df.head(10))

df.to_csv("pccomponentes_gaming_scraping_raw.csv", index=False)

Scrapeando: PC (https://www.pccomponentes.com/ordenadores-gaming)
  → 45 productos encontrados en PC
Scrapeando: Monitores (https://www.pccomponentes.com/monitores-gaming)
  → 40 productos encontrados en Monitores
Scrapeando: Teclados (https://www.pccomponentes.com/teclados-gaming)
  → 40 productos encontrados en Teclados
Scrapeando: Ratones (https://www.pccomponentes.com/ratones-gaming)
  → 40 productos encontrados en Ratones
Scrapeando: Cascos (https://www.pccomponentes.com/auriculares-gaming)
  → 40 productos encontrados en Cascos
Scrapeando: Mandos (https://www.pccomponentes.com/mandos-pc)
  → 40 productos encontrados en Mandos

✅ Total de productos scrapeados: 245
✅ Subcategorías únicas: <StringArray>
['PC', 'Monitores', 'Teclados', 'Ratones', 'Cascos', 'Mandos']
Length: 6, dtype: str

Primeras filas:
        sku                                             nombre  marca  \
0  10880191  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   
1  11027737  PcCom Imperial AMD Ryze

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ====== RUTA DE SALIDA ======
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "pc componentes scrapping limpio.csv"

# ====== CARGA DEL SCRAPING BRUTO ======
# Cambia el nombre si tu CSV bruto tiene otro nombre
input_file = output_dir / "pccomponentes_gaming_scraping_raw.csv"
df = pd.read_csv(input_file)

# ====== FUNCIONES DE LIMPIEZA ======
def limpiar_url(url):
    if pd.isna(url) or url is None:
        return np.nan
    url = str(url).strip()
    m = re.search(r"https?://[^\)\]\s]+", url)
    return m.group(0) if m else url

def limpiar_texto(x):
    if pd.isna(x) or x is None:
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else np.nan

def normalizar_numero(x):
    if pd.isna(x) or x is None:
        return np.nan
    x = str(x).strip()
    x = x.replace(".", "").replace(",", ".")
    m = re.search(r"\d+(\.\d+)?", x)
    return float(m.group()) if m else np.nan

def normalizar_entero(x):
    if pd.isna(x) or x is None:
        return np.nan
    x = str(x)
    m = re.search(r"\d+", x.replace(".", ""))
    return int(m.group()) if m else np.nan

# ====== LIMPIEZA GENERAL ======
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)

# URLs
if "product_url" in df.columns:
    df["product_url"] = df["product_url"].apply(limpiar_url)

if "url_categoria" in df.columns:
    df["url_categoria"] = df["url_categoria"].apply(limpiar_url)

# Normalizar numéricos
for col in ["precio_actual", "precio_original", "descuento_pct", "rating", "num_opiniones"]:
    if col in df.columns:
        df[col] = df[col].apply(normalizar_numero)

if "sku" in df.columns:
    df["sku"] = df["sku"].apply(normalizar_entero)

# ====== QUITAR DUPLICADOS ======
subset_cols = [c for c in ["sku", "product_url", "nombre"] if c in df.columns]
if subset_cols:
    df = df.drop_duplicates(subset=subset_cols, keep="first").copy()
else:
    df = df.drop_duplicates(keep="first").copy()

# ====== QUITAR FILAS VACÍAS/INÚTILES ======
# Conserva rating y opiniones, pero elimina filas que no tengan ni nombre ni URL
cond_basica = pd.Series(True, index=df.index)

if "nombre" in df.columns:
    cond_basica &= df["nombre"].notna()

if "product_url" in df.columns:
    cond_basica &= df["product_url"].notna()

df = df[cond_basica].copy()

# Elimina filas totalmente vacías
df = df.dropna(how="all").copy()

# ====== LIMPIEZA DE VALORES MALOS ======
if "precio_actual" in df.columns:
    df.loc[df["precio_actual"] <= 0, "precio_actual"] = np.nan

if "precio_original" in df.columns:
    df.loc[df["precio_original"] <= 0, "precio_original"] = np.nan

if "descuento_pct" in df.columns:
    df.loc[(df["descuento_pct"] < 0) | (df["descuento_pct"] > 90), "descuento_pct"] = np.nan

if "rating" in df.columns:
    df.loc[(df["rating"] < 0) | (df["rating"] > 5), "rating"] = np.nan

if "num_opiniones" in df.columns:
    df.loc[df["num_opiniones"] < 0, "num_opiniones"] = np.nan

# ====== ORDEN DE COLUMNAS ======
cols_priority = [
    "sku", "nombre", "marca", "subcategoria", "categoria_raw",
    "precio_actual", "precio_original", "descuento_pct",
    "rating", "num_opiniones",
    "seller_raw", "promo_raw",
    "product_url", "url_categoria"
]

cols_existing = [c for c in cols_priority if c in df.columns]
other_cols = [c for c in df.columns if c not in cols_existing]
df = df[cols_existing + other_cols].copy()

# ====== GUARDAR CSV LIMPIO ======
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {output_file}")
print(f"Filas finales: {len(df)}")
print(f"Columnas finales: {list(df.columns)}")

Archivo guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\pc componentes scrapping limpio.csv
Filas finales: 231
Columnas finales: ['sku', 'nombre', 'marca', 'subcategoria', 'categoria_raw', 'precio_actual', 'precio_original', 'descuento_pct', 'rating', 'num_opiniones', 'seller_raw', 'promo_raw', 'product_url', 'url_categoria', 'precio_actual_raw', 'precio_original_raw', 'descuento_pct_raw', 'rating_raw', 'opiniones_raw']
